# 🏎️ DENSO VisionMind — Kaggle GPU Benchmark Notebook for Surya Vision Suite
Notebook này được tối ưu hóa 100% để chạy trên môi trường **Kaggle GPU (T4 / P100)** nhằm đánh giá và bóc tách toàn bộ **53 tài liệu PDF nhà máy DENSO** (`COBOTTA_PRO_E.pdf`, `kanban.pdf`, `saitekikeiro.pdf`,...).

### 📌 Hướng dẫn tải lên Kaggle:
1. Mở Kaggle Notebook -> Click **New Notebook**.
2. Bật GPU Accelerator: **Settings -> Accelerator -> GPU T4 x2**.
3. Upload thư mục `data/documents/documents` làm **Kaggle Dataset** (tên gợi ý: `denso-factory-documents`).
4. Import và chạy toàn bộ notebook này.

In [ ]:
# 1. Cài đặt các thư viện cần thiết trên môi trường Kaggle
!pip install -q surya-ocr pymupdf Pillow matplotlib pandas tqdm Levenshtein

import os
import time
import json
import torch
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import fitz  # PyMuPDF

print(f"⚡ PyTorch Version: {torch.__version__}")
print(f"🔥 GPU Status: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (Cần bật GPU trên Kaggle!)'}")

## 2. Nạp Mô Hình Surya FastLayoutPredictor & RecognitionPredictor trên GPU

In [ ]:
from surya.fast_layout import FastLayoutPredictor
from surya.recognition import RecognitionPredictor

print("🚀 Đang khởi tạo Surya Layout Predictor trên Kaggle GPU...")
layout_model = FastLayoutPredictor()

print("🚀 Đang khởi tạo Surya Recognition (OCR) Predictor...")
ocr_model = RecognitionPredictor()

print("✅ Bộ mô hình Surya đã sẵn sàng tăng tốc xử lý hàng loạt trên Kaggle!")

## 3. Cấu hình Thư mục Đầu vào chứa 53 File PDF DENSO

In [ ]:
# Đường dẫn thư mục dữ liệu trên Kaggle hoặc Local
KAGLE_INPUT_DIR = Path("/kaggle/input/denso-factory-documents")
LOCAL_INPUT_DIR = Path("../data/documents/documents")  # Hoặc đường dẫn local của bạn

DATASET_DIR = KAGLE_INPUT_DIR if KAGLE_INPUT_DIR.exists() else LOCAL_INPUT_DIR

pdf_files = list(DATASET_DIR.glob("*.pdf"))
print(f"📁 Tìm thấy tổng cộng {len(pdf_files)} file PDF tài liệu DENSO:")
for idx, f in enumerate(pdf_files[:10], 1):
    print(f"  {idx}. {f.name} ({f.stat().st_size / 1024 / 1024:.2f} MB)")
if len(pdf_files) > 10:
    print(f"  ... và {len(pdf_files)-10} file PDF khác.")

## 4. Tiến Trình Bóc Tách Layout Hàng Loạt (Batch Layout & Bounding Box Extraction)

In [ ]:
def process_pdf_surya(pdf_path, max_pages=3):
    """Trích xuất Layout & OCR trên tối đa max_pages trang đầu của từng file PDF"""
    results = []
    try:
        doc = fitz.open(pdf_path)
        total_p = min(len(doc), max_pages)
        
        for p_idx in range(total_p):
            page = doc[p_idx]
            pix = page.get_pixmap(dpi=150)
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            
            # Dự đoán Layout Bounding Box bằng Surya
            layout_pred = layout_model([img])[0]
            blocks = layout_pred.bboxes
            
            try:
                ocr_pred = ocr_model([img], [layout_pred])[0]
                if hasattr(ocr_pred, 'blocks') and ocr_pred.blocks:
                    blocks = ocr_pred.blocks
            except Exception:
                pass
                
            for b_idx, b in enumerate(blocks):
                label = getattr(b, 'label', 'paragraph').lower()
                bbox = b.bbox
                norm_bbox = [
                    int((bbox[0]/pix.width)*1000),
                    int((bbox[1]/pix.height)*1000),
                    int((bbox[2]/pix.width)*1000),
                    int((bbox[3]/pix.height)*1000)
                ]
                txt = getattr(b, 'html', '') or getattr(b, 'text', '') or f"[{label.upper()}]"
                
                results.append({
                    "file_name": pdf_path.name,
                    "page_number": p_idx + 1,
                    "block_id": b_idx + 1,
                    "label": label,
                    "bbox_raw": [int(v) for v in bbox],
                    "bbox_norm_1000": norm_bbox,
                    "text_snippet": str(txt)[:100]
                })
        doc.close()
    except Exception as e:
        print(f"❌ Lỗi khi đọc file {pdf_path.name}: {e}")
    return results

# Chạy tiến trình trích xuất trên toàn bộ danh sách PDF DENSO
all_extracted_records = []
start_time = time.time()

print("⏳ Đang thực thi bóc tách 53 file PDF DENSO trên Kaggle GPU...")
for pdf in tqdm(pdf_files):
    rec = process_pdf_surya(pdf, max_pages=3)
    all_extracted_records.extend(rec)

total_time = time.time() - start_time
print(f"\n🎉 HOÀN THÀNH BÓC TÁCH!")
print(f"⚡ Tổng thời gian xử lý: {total_time:.2f} giây (~{total_time/max(1, len(pdf_files)):.2f}s / file PDF)")
print(f"📊 Tổng số Bounding Box phát hiện: {len(all_extracted_records)} bboxes")

## 5. Tổng Hợp Kết Quả Báo Cáo & Xuất File CSV / JSON

In [ ]:
# Chuyển đổi kết quả sang Pandas DataFrame
df_extracted = pd.DataFrame(all_extracted_records)

if not df_extracted.empty:
    # Tổng hợp số lượng từng loại phần tử bóc tách được
    summary_counts = df_extracted['label'].value_counts().reset_index()
    summary_counts.columns = ['Layout Category', 'Detected Count']
    
    print("\n" + "="*60)
    print("🏆 PHÂN PHỐI BỐ CỤC PHÁT HIỆN TỪ TÀI LIỆU DENSO (SURYA VISION)")
    print("="*60)
    display(summary_counts)
    
    # Xuất file CSV kết quả bóc tách
    output_csv = "denso_surya_extracted_layout.csv"
    df_extracted.to_csv(output_csv, index=False)
    print(f"\n💾 File kết quả CSV đã lưu tại: {output_csv}")
else:
    print("⚠️ Chưa có dữ liệu trích xuất. Hãy kiểm tra lại đường dẫn file PDF đầu vào!")